Create Patient Radiology Embeddings per Visit

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A10-24Q
VRAM (GB): 25.769345024


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

# ------------------------------
# CONFIG
# ------------------------------
CHUNK_SIZE = 10_000
INITIAL_BATCH_SIZE = 8
MODEL_NAME = "google/medgemma-4b-pt"
MAX_LEN = 512

# Paths
MIMIC_NOTES_PATH = r".....mimic iv\note\radiology.csv.gz"
DIAGNOSES_PATH = r"........mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"
OUTPUT_PATH = r".....cancer_admission_embs_radiology.npy"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ------------------------------
# TARGET ICD CODES (SOLID CANCERS)
# ------------------------------
TARGET_ICD9_PREFIXES = (
    "140", "141", "142", "143", "144", "145", "146", "147", "148", "149",
    "153", "154",
    "162",
    "174",
    "185",
    "188"
)

TARGET_ICD10_PREFIXES = (
    "C00", "C01", "C02", "C03", "C04", "C05", "C06", "C07", "C08",
    "C18", "C19", "C20",
    "C34",
    "C50",
    "C61",
    "C67"
)

# ------------------------------
# DEVICE & MODEL
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

print("Loading MedGemma model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()
print("Model loaded.\n")

# ------------------------------
# EMBEDDING FUNCTIONS
# ------------------------------
@torch.inference_mode()
def embed_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs)
    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    hidden = outputs.last_hidden_state

    emb = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)

    # bf16 → fp32 for NumPy compatibility
    return emb.float().cpu().numpy()


def safe_embed_batch(texts):
    batch = texts
    while len(batch) > 0:
        try:
            return embed_batch(batch)
        except RuntimeError as e:
            if "out of memory" in str(e):
                torch.cuda.empty_cache()
                batch = batch[: len(batch) // 2]
                print(f"OOM → reducing batch to {len(batch)}")
            else:
                raise
    return None

# ------------------------------
# LOAD TARGET DIAGNOSES
# ------------------------------
print("Loading cancer diagnoses...")
diag = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"]
)

diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()

target_mask = (
    ((diag.icd_version == 9) & diag.icd_code.str.startswith(TARGET_ICD9_PREFIXES)) |
    ((diag.icd_version == 10) & diag.icd_code.str.startswith(TARGET_ICD10_PREFIXES))
)

target_diag = diag[target_mask]

target_hadm_ids = set(target_diag["hadm_id"].unique())
target_subject_ids = set(target_diag["subject_id"].unique())

print(f"Found {len(target_subject_ids)} cancer patients with "
      f"{len(target_hadm_ids)} admissions.\n")

# ------------------------------
# AGGREGATE RADIOLOGY NOTES PER ADMISSION
# ------------------------------
admission_notes = defaultdict(list)
admission_subject = {}

for chunk_idx, chunk in enumerate(
    pd.read_csv(
        MIMIC_NOTES_PATH,
        usecols=["subject_id", "hadm_id", "text"],
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    chunk = chunk.dropna(subset=["subject_id", "hadm_id", "text"])
    chunk = chunk[chunk["hadm_id"].isin(target_hadm_ids)]

    for sid, hid, text in zip(chunk.subject_id, chunk.hadm_id, chunk.text):
        text = str(text).strip()
        if len(text) < 20:
            continue
        admission_notes[hid].append(text)
        admission_subject[hid] = sid

    print(f"Processed chunk {chunk_idx}, admissions collected: {len(admission_notes)}")

print(f"\nTotal cancer admissions with radiology notes: {len(admission_notes)}\n")

# ------------------------------
# COMPUTE ADMISSION-LEVEL EMBEDDINGS
# ------------------------------
admission_embeddings = []
admission_ids = []
subject_ids = []

for i, (hid, notes) in enumerate(admission_notes.items(), start=1):
    embs = []

    for j in range(0, len(notes), INITIAL_BATCH_SIZE):
        batch = notes[j : j + INITIAL_BATCH_SIZE]
        batch_emb = safe_embed_batch(batch)
        if batch_emb is not None:
            embs.append(batch_emb)

    if not embs:
        continue

    admission_emb = np.vstack(embs).mean(axis=0)

    if np.isnan(admission_emb).any():
        continue

    admission_embeddings.append(admission_emb)
    admission_ids.append(hid)
    subject_ids.append(admission_subject[hid])

    if i % 50 == 0:
        print(f"Embedded {i}/{len(admission_notes)} admissions")

# ------------------------------
# SAVE OUTPUTS
# ------------------------------
X = np.vstack(admission_embeddings)
hadm_ids = np.array(admission_ids)
subject_ids = np.array(subject_ids)

np.save(OUTPUT_PATH, X)
np.save(OUTPUT_PATH.replace(".npy", "_hadm_ids.npy"), hadm_ids)
np.save(OUTPUT_PATH.replace(".npy", "_subject_ids.npy"), subject_ids)

print(f"\nSaved {len(hadm_ids)} admission embeddings")
print("Embedding shape:", X.shape)
